# 1B.4 Pandas use cases

Hieronder gaan we 3 use cases overlopen, maar eerst nemen we een korte kijk naar de data. Merk op dat we voor bepaalde objecten zoals ```DataFrame``` uit Pandas geen ```print()``` gebruiken omdat Jupyter notebooks automatisch tabulaire data mooi zal vormgeven zoals hieronder getoond (het laatste statement wordt altijd afgedrukt als output van de cel):

In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv('COVID19BE_CASES_AGESEX.csv') # lees de data in
df.head(10) # toont de eerste 10 rijen

,DATE,PROVINCE,REGION,AGEGROUP,SEX,CASES
0,2020-03-01,Antwerpen,Flanders,40-49,M,1
1,2020-03-01,Brussels,Brussels,10-19,F,1
2,2020-03-01,Brussels,Brussels,10-19,M,1
3,2020-03-01,Brussels,Brussels,20-29,M,1
4,2020-03-01,Brussels,Brussels,30-39,F,1
5,2020-03-01,Brussels,Brussels,40-49,F,1
6,2020-03-01,Brussels,Brussels,50-59,M,1
7,2020-03-01,Liège,Wallonia,40-49,M,3
8,2020-03-01,Limburg,Flanders,70-79,M,1
9,2020-03-01,OostVlaanderen,Flanders,50-59,F,1


## Use case 1

Verwijder alle rijen met NaNs uit een dataframe.

Hiervoor gaan we 2 manieren gebruiken: 
- 1: Zelf doorlopen van de rijen
- 2: Ingebouwde Pandas functie

#### 1): de rijen doorlopen en per cel kijken of er ```nan``` of ```NaN``` waarden inzitten:

In [19]:
rijen_met_nans = []
for r, rij in df.iterrows(): # ga door de rijen, 'r' is het rijnummer
    for cell in rij:
        if cell in {np.nan}: # met oudere versies van Numpy gebruik je {np.nan, np.NaN}:
            rijen_met_nans.append(r)
            break

Zoals eerder gezien is het niet aangewezen om ```np.nan``` en ```np.NaN``` als waarden te aanschouwen. Beter is om de ingebouwde ```isnull()``` methode te gebruiken in combinatie met ```any()``` die telt of er meer dan 0 ```nan``` aanwezig zijn:

In [20]:
rijen_met_nans = []
for r, rij in df.iterrows():
    if rij.isnull().any():
        rijen_met_nans.append(r)

In [21]:
print('Lengte dataframe met NaNs:', len(df))
df_zonder_nans = df.drop(rijen_met_nans)
print('Lengte zonder NaNs:', len(df_zonder_nans))

Lengte dataframe met NaNs: 66175
Lengte zonder NaNs: 59923


#### 2): we kunnen ook gewoon de ```dropna()``` methode oproepen:

In [22]:
print('Lengte zonder NaNs:', len(df.dropna()))

Lengte zonder NaNs: 59923


## Use case 2

Verwijder alle rijen waarbij de numerische waarde groter is dan 3 standaarddeviaties van het gemiddelde voor een bepaalde kolom.

We vergelijken de absolute waarden van de kolom CASES met de standaarddeviatie berekend via de Pandas Series-methode ```std()``` (steekproef-standaardafwijking) alsook het gemiddelde met ```mean()```:

In [23]:
std =  df['CASES'].std()
gem = df['CASES'].mean()
gem, std

(np.float64(12.433713638080846), np.float64(25.317462390772956))

Nu nemen we de absolute waarde van de waarde van de 'CASES' variabele min het gemiddelde van die variabele (dit levert ons een Series-object op):

In [24]:
absol = abs(df['CASES'] - gem)
absol

0        11.433714
1        11.433714
2        11.433714
3        11.433714
4        11.433714
           ...    
66170     9.433714
66171    11.433714
66172    11.433714
66173     9.433714
66174    11.433714
Name: CASES, Length: 66175, dtype: float64

Ten slotte gebruiken we `std` en `absol` in een filter:

In [25]:
print('Lengte dataframe met afwijkende waarden:', len(df))
df_zonder_afwijkende_waarden = df[absol < std * 3]
print('Lengte dataframe zonder afwijkende waarden:', len(df_zonder_afwijkende_waarden))

Lengte dataframe met afwijkende waarden: 66175
Lengte dataframe zonder afwijkende waarden: 64870


## Use case 3

Verwijder alle rijen voor provincies die in minder dan 8% van de gevallen voorkomen.

Aanwezigheden in dataframe:

In [26]:
df['PROVINCE'].value_counts(normalize=True)

PROVINCE
Antwerpen         0.107718
Brussels          0.105345
OostVlaanderen    0.098950
WestVlaanderen    0.098918
Hainaut           0.098014
Liège             0.093977
VlaamsBrabant     0.093218
Limburg           0.091167
Namur             0.075892
BrabantWallon     0.072469
Luxembourg        0.064331
Name: proportion, dtype: float64

Vermits ```value_counts()``` gewoon een ```Series``` object teruggeeft, kunnen we het ook filteren:

In [27]:
value_counts = df['PROVINCE'].value_counts(normalize=True)
print(value_counts)

PROVINCE
Antwerpen         0.107718
Brussels          0.105345
OostVlaanderen    0.098950
WestVlaanderen    0.098918
Hainaut           0.098014
Liège             0.093977
VlaamsBrabant     0.093218
Limburg           0.091167
Namur             0.075892
BrabantWallon     0.072469
Luxembourg        0.064331
Name: proportion, dtype: float64


In [28]:
te_behouden = value_counts[value_counts > 0.08]
te_behouden

PROVINCE
Antwerpen         0.107718
Brussels          0.105345
OostVlaanderen    0.098950
WestVlaanderen    0.098918
Hainaut           0.098014
Liège             0.093977
VlaamsBrabant     0.093218
Limburg           0.091167
Name: proportion, dtype: float64

De output is een ```Series``` met de provincies als index, die kunnen we even omzetten naar een lijst om te visualiseren:

In [29]:
lijst_frequente_provincies = list(te_behouden.index)
lijst_frequente_provincies

['Antwerpen',
 'Brussels',
 'OostVlaanderen',
 'WestVlaanderen',
 'Hainaut',
 'Liège',
 'VlaamsBrabant',
 'Limburg']

Nu kunnen we deze lijst combineren met de ```isin()``` methode van ```Series``` en zo alle provincies die wel in de te behouden index zitten kunnen bijgehouden worden:

In [30]:
df_zonder_infrequente_provincies = df[df['PROVINCE'].isin(te_behouden.index)]
df_zonder_infrequente_provincies['PROVINCE'].value_counts(normalize=True)

PROVINCE
Antwerpen         0.136819
Brussels          0.133804
OostVlaanderen    0.125682
WestVlaanderen    0.125641
Hainaut           0.124492
Liège             0.119365
VlaamsBrabant     0.118401
Limburg           0.115796
Name: proportion, dtype: float64

Hopelijk illustreerden deze voorbeelden dat we met Jupyter notebooks heel makkelijk Python code kunnen combineren met uitleg en visualisatie van data. Je vindt er verschillende op het internet (denk bv. aan de voorbeelden die hier te vinden zijn: https://github.com/jupyter/jupyter/wiki/A-gallery-of-interesting-Jupyter-Notebooks#statistics-machine-learning-and-data-science)